# Convolución 2D con kernel 3x3

**Ejercicio académico — Procesamiento de Imágenes**

En este notebook se:
1. Carga una imagen (en **RGB** o en **escala de grises**).
2. Define un **kernel (filtro) de 3x3** ingresado por el usuario.
3. Aplica la **convolución 2D** de forma manual (sin `scipy` ni `cv2`), para entender paso a paso cómo funciona.
4. Muestra un **subplot** con la imagen original y la imagen filtrada.

---

## ¿Qué es la convolución?

La convolución 2D es una operación que, para cada píxel de la imagen:
1. Toma la **vecindad de 3x3 píxeles** centrada en ese píxel.
2. **Multiplica** cada píxel de esa vecindad por el valor correspondiente del kernel.
3. **Suma** los 9 resultados (y opcionalmente divide entre un factor de normalización).
4. Ese resultado es el **nuevo valor del píxel** en la imagen de salida.

En los **bordes** de la imagen la vecindad se sale de los límites. La solución más simple (la que usamos aquí) es **rellenar con ceros** (*zero padding*).

In [ ]:
# Librerías necesarias
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

%matplotlib inline

## 1. Función para cargar la imagen

La función carga la imagen en el modo solicitado:
- `'gris'` → escala de grises (1 canal, modo `'L'` de PIL).
- `'rgb'` → color (3 canales).

Devuelve el array de la imagen como `float32` (para no perder precisión en la convolución) **normalizado al rango `[0, 1]`**, y una bandera `es_gris` que indica el modo.

In [ ]:
def cargar_imagen(ruta, modo="gris"):
    """
    Carga una imagen desde `ruta`.
    modo: 'gris' o 'rgb'.
    Retorna (array float32 en [0,1], es_gris: bool).
    """
    img = Image.open(ruta)
    if modo == "gris":
        img = img.convert("L")          # 1 canal
        es_gris = True
    elif modo == "rgb":
        img = img.convert("RGB")        # 3 canales
        es_gris = False
    else:
        raise ValueError("modo debe ser 'gris' o 'rgb'")
    return np.asarray(img, dtype=np.float32) / 255.0, es_gris

print("Función cargar_imagen definida.")

## 2. Función de convolución 2D (implementación manual)

Recorremos cada píxel y calculamos la suma ponderada con el kernel:
- Se hace **padding con ceros** de 1 píxel alrededor para manejar los bordes.
- Si la imagen es **RGB**, se aplica la convolución **canal por canal** (el mismo kernel a R, G y B por separado).

In [ ]:
def convolucion_3x3(imagen, kernel):
    """
    Aplica convolución 2D con un kernel 3x3 usando zero padding.
    imagen: array 2D (gris) o 3D (RGB) float.
    kernel: array 3x3 float.
    """
    kernel = np.asarray(kernel, dtype=np.float32)
    if kernel.shape != (3, 3):
        raise ValueError("El kernel debe ser de 3x3")

    if imagen.ndim == 2:                      # escala de grises
        pad = np.pad(imagen, 1, mode="constant")   # padding de 1 píxel con ceros
        H, W = imagen.shape
        salida = np.zeros_like(imagen)
        for i in range(H):
            for j in range(W):
                vecindad = pad[i:i+3, j:j+3]   # vecindad 3x3 centrada en (i, j)
                salida[i, j] = np.sum(vecindad * kernel)
        return salida

    elif imagen.ndim == 3:                    # RGB: convolución por canal
        canales = [convolucion_3x3(imagen[:, :, c], kernel) for c in range(imagen.shape[2])]
        return np.stack(canales, axis=2)

    else:
        raise ValueError("La imagen debe ser 2D (gris) o 3D (RGB)")

print("Función convolucion_3x3 definida.")

## 3. Entrada del usuario: kernel 3x3

El usuario ingresa los 9 valores del kernel **fila por fila** (valores separados por espacios). Además puede ingresar un **factor de normalización** (útil para filtros de suavizado).

**Kernels de ejemplo para probar:**

| Efecto | Kernel | Normalización |
|---|---|---|
| Suavizado (blur) | `1 1 1` / `1 1 1` / `1 1 1` | 9 |
| Enfoque (sharpen) | `0 -1 0` / `-1 5 -1` / `0 -1 0` | 1 |
| Detección de bordes | `-1 -1 -1` / `-1 8 -1` / `-1 -1 -1` | 1 |
| Sobel vertical | `-1 0 1` / `-2 0 2` / `-1 0 1` | 1 |

In [ ]:
# === ENTRADA DEL USUARIO ===
kernel = np.zeros((3, 3), dtype=np.float32)
print("Ingrese el kernel 3x3 fila por fila (9 valores separados por espacios):")
for f in range(3):
    fila = [float(x) for x in input(f"  Fila {f+1}: ").split()]
    if len(fila) != 3:
        raise ValueError("Debe ingresar exactamente 3 valores por fila")
    kernel[f, :] = fila

norm = input("Factor de normalización (Enter = 1, no normaliza): ").strip()
norm = float(norm) if norm else 1.0
kernel = kernel / norm

print("\nKernel usado:")
print(kernel)

## 4. Cargar la imagen y aplicar la convolución

Cambie `ruta_imagen` y `modo` (`"gris"` o `"rgb"`) según la imagen que quiera probar. Si el archivo no existe, se genera automáticamente una **imagen sintética de prueba** (un círculo con gradiente) para que el ejercicio funcione igual.

In [ ]:
# === CONFIGURACIÓN DE LA IMAGEN ===
ruta_imagen = "imagen.jpg"   # cambie por la ruta de su imagen
modo = "gris"                # "gris" o "rgb"

try:
    img, es_gris = cargar_imagen(ruta_imagen, modo)
except FileNotFoundError:
    print(f"No se encontró '{ruta_imagen}'. Se genera una imagen sintética de prueba.")
    H, W = 256, 256
    yy, xx = np.mgrid[0:H, 0:W]
    img = ((xx - 128)**2 + (yy - 128)**2 < 80**2).astype(np.float32)
    img = img * (0.5 + 0.5 * xx / W)          # gradiente horizontal
    img = np.stack([img] * 3, axis=2)         # la hacemos RGB
    es_gris = (modo == "gris")
    if es_gris:
        img = img[:, :, 0]

print(f"Imagen cargada. Dimensiones: {img.shape}, escala de grises: {es_gris}")

# Aplicar la convolución
img_filtrada = convolucion_3x3(img, kernel)
print("Convolución aplicada.")

## 5. Visualización: imagen original vs. imagen filtrada

Usamos `plt.subplots(1, 2)` para mostrarlas **lado a lado**. Si la imagen es gris usamos el mapa de colores `gray`. Se usa `np.clip` para llevar los valores que quedaron fuera de `[0, 1]` de vuelta al rango visible.

In [ ]:
cmap = "gray" if es_gris else None

fig, ejes = plt.subplots(1, 2, figsize=(12, 6))

ejes[0].imshow(img, cmap=cmap, vmin=0, vmax=1)
ejes[0].set_title("Imagen original")
ejes[0].axis("off")

ejes[1].imshow(np.clip(img_filtrada, 0, 1), cmap=cmap, vmin=0, vmax=1)
ejes[1].set_title("Imagen filtrada (kernel 3x3)")
ejes[1].axis("off")

plt.tight_layout()
plt.show()

## Conclusiones

- La **convolución 3x3** modifica cada píxel según su vecindad, lo que permite implementar los filtros clásicos del procesamiento de imágenes.
- Con un **kernel de promedio** (todos 1, normalizado por 9) la imagen se **suaviza / desenfoca**.
- Con el kernel de **enfoque** (centro 5, vecinos -1) la imagen gana **nitidez**.
- Con kernels tipo **Sobel o Laplaciano** se **detectan bordes**: las zonas planas se oscurecen y los cambios bruscos de intensidad se resaltan.
- El **zero padding** genera bordes oscuros artificiales; alternativas comunes son replicar o reflejar los bordes.
- Para imágenes **RGB**, la convolución se aplica por separado a cada canal, manteniendo la estructura del color.

---

# Ejercicio 2: Doble convolución (filtrar 2 veces con kernels 3x3 diferentes)

Ahora se toma la misma imagen y se le aplican **dos convoluciones encadenadas**:

1. **Primera convolución** con un kernel 3x3 → imagen intermedia.
2. **Segunda convolución** con otro kernel 3x3, aplicada sobre el **resultado de la primera** → imagen final.

Esto permite construir efectos compuestos. Por ejemplo:
- **Blur + Blur** = desenfoque más fuerte.
- **Blur + Enfoque** = casi la imagen original (los filtros tienden a compensarse).
- **Sobel horizontal + Sobel vertical** = combinación base de detección de bordes.
- **Blur + Bordes** = detección de bordes con menos ruido (los bordes salen más limpios).

Al final se muestra un subplot con **3 imágenes**: original, resultado de la 1ª convolución y resultado de la 2ª convolución.

> Nota: el `zero padding` de la primera convolución mete un borde oscuro artificial; si en la segunda convolución se usa un filtro de bordes, ese borde artificial puede aparecer resaltado. Es un comportamiento esperado.

## 6. Entrada del usuario: los 2 kernels 3x3

Se piden al usuario los **dos kernels** (mismo formato que en el ejercicio 1: 3 filas de 3 valores separados por espacios, más un factor de normalización). El kernel de la segunda convolución se aplica sobre el resultado de la primera.

In [ ]:
# === ENTRADA DE LOS 2 KERNELS (EJERCICIO 2) ===

def pedir_kernel(nombre):
    """Pide al usuario un kernel 3x3 y su factor de normalización."""
    k = np.zeros((3, 3), dtype=np.float32)
    print(f"--- Kernel {nombre} (fila por fila, 3 valores separados por espacios) ---")
    for f in range(3):
        fila = [float(x) for x in input(f"  Fila {f+1}: ").split()]
        if len(fila) != 3:
            raise ValueError("Debe ingresar exactamente 3 valores por fila")
        k[f, :] = fila
    norm = input("  Factor de normalización (Enter = 1): ").strip()
    return k / (float(norm) if norm else 1.0)

kernel1 = pedir_kernel("1 (primera convolución)")
print()
kernel2 = pedir_kernel("2 (segunda convolución)")

print("\nKernel 1:"); print(kernel1)
print("\nKernel 2:"); print(kernel2)

## 7. Aplicar las 2 convoluciones encadenadas

Se aplica la primera convolución a la imagen original y, sobre ese resultado, la segunda convolución. Se reutilizan las funciones `cargar_imagen` y `convolucion_3x3` ya definidas arriba.

In [ ]:
# === APLICAR LAS 2 CONVOLUCIONES (EJERCICIO 2) ===
# Reutiliza img, es_gris y convolucion_3x3 del ejercicio 1 (ejecuta las celdas anteriores primero)

# 1ª convolución: sobre la imagen original
img_conv1 = convolucion_3x3(img, kernel1)

# 2ª convolución: sobre el resultado de la primera
img_conv2 = convolucion_3x3(img_conv1, kernel2)

print(f"Imagen original:        {img.shape}")
print(f"Tras 1ª convolución:    {img_conv1.shape}")
print(f"Tras 2ª convolución:    {img_conv2.shape}")
print("Doble convolución aplicada.")

## 8. Visualización: original → 1ª convolución → 2ª convolución

Subplot con **3 columnas**: la imagen original, el resultado intermedio (1er kernel) y el resultado final (2º kernel). Así se ve claramente el efecto acumulado de los dos filtros.

In [ ]:
# === VISUALIZACIÓN DE LAS 3 IMÁGENES (EJERCICIO 2) ===

titulos = ["Imagen original",
           "1ª convolución (kernel 1)",
           "2ª convolución (kernel 2)"]
imagenes = [img, np.clip(img_conv1, 0, 1), np.clip(img_conv2, 0, 1)]

fig, ejes = plt.subplots(1, 3, figsize=(16, 6))

for eje, imagen, titulo in zip(ejes, imagenes, titulos):
    eje.imshow(imagen, cmap=cmap, vmin=0, vmax=1)
    eje.set_title(titulo)
    eje.axis("off")

plt.tight_layout()
plt.show()

### Conclusiones del ejercicio 2

- Aplicar **dos convoluciones encadenadas** equivale a convolucionar con un único kernel compuesto (propiedad asociativa de la convolución): `k2 ∗ (k1 ∗ imagen) = (k2 ∗ k1) ∗ imagen`.
- **Blur + Blur** produce un desenfoque más fuerte que uno solo (la 'ventana' efectiva crece).
- **Blur + Detección de bordes** da bordes más limpios, porque el suavizado previo elimina ruido de alta frecuencia.
- Los kernels negativos (Sobel, Laplaciano) generan valores fuera de `[0, 1]`; por eso se usa `np.clip` al visualizar.
- Cada convolución extra hace que el borde de ceros del padding 'entre' más en la imagen, oscureciendo los márgenes.

---

# Ejercicio 3: Bloque convolucional tipo CNN (3 kernels → ReLU → Max Pooling 2x2)

Este ejercicio simula, de forma manual, el **primer bloque de una red neuronal convolucional (CNN)**:

1. **3 convoluciones** con 3 kernels 3x3 **diferentes** (no se piden al usuario: se usan kernels ya establecidos, con bordes, texturas y desenfoque).
2. Después de **cada convolución** se aplica **ReLU**: `f(x) = max(0, x)` — elimina los valores negativos, igual que hace una neurona en una CNN.
3. Al final se aplica **Max Pooling 2x2**: se divide el mapa de características en bloques de 2x2 y de cada bloque se conserva solo el **valor máximo**, reduciendo el tamaño a la mitad (H/2 x W/2) y quedándose con las activaciones más fuertes.

El resultado son **3 mapas de características** que resaltan diferentes patrones de la imagen (bordes horizontales, verticales y suavizado), tal como ocurre en las primeras capas de una CNN.

In [ ]:
## 9. Los 3 kernels 3x3 (ya establecidos, no se piden al usuario)

Se usan 3 kernels clásicos, cada uno especializado en detectar un patrón distinto:

- **Kernel 1 — Sobel vertical**: detecta **bordes verticales** (cambios de intensidad de izquierda a derecha).
- **Kernel 2 — Sobel horizontal**: detecta **bordes horizontales** (cambios de intensidad de arriba hacia abajo).
- **Kernel 3 — Blur (promedio)**: **suaviza** la imagen, resaltando zonas de baja frecuencia.

Al normalizar por su suma absoluta (2 o 9) se evita que las salidas se saturen.

In [ ]:
# === LOS 3 KERNELS YA ESTABLECIDOS (EJERCICIO 3) ===

kernels_ej3 = {
    "Kernel 1: Sobel vertical (bordes verticales)":
        np.array([[-1, 0, 1],
                  [-2, 0, 2],
                  [-1, 0, 1]], dtype=np.float32) / 8.0,

    "Kernel 2: Sobel horizontal (bordes horizontales)":
        np.array([[-1, -2, -1],
                  [ 0,  0,  0],
                  [ 1,  2,  1]], dtype=np.float32) / 8.0,

    "Kernel 3: Blur (promedio 3x3)":
        np.ones((3, 3), dtype=np.float32) / 9.0,
}

for nombre, k in kernels_ej3.items():
    print(f"{nombre}:\n{k}\n")

## 10. Funciones ReLU y Max Pooling 2x2

- **ReLU** (`np.maximum(0, x)`): se aplica **elemento a elemento** sobre el mapa de características tras cada convolución. Convierte todos los negativos en 0, introduciendo la no-linealidad.
- **Max Pooling 2x2** (paso 2): recorre el mapa en ventanas de 2x2 **sin solaparse** y toma el máximo de cada ventana. La salida tiene la **mitad del alto y la mitad del ancho**. Si alguna dimensión es impar, se descarta la última fila/columna. Se aplica canal por canal si la imagen es RGB.

In [ ]:
# === FUNCIONES ReLU Y MAX POOLING (EJERCICIO 3) ===

def relu(x):
    """ReLU elemento a elemento: f(x) = max(0, x)."""
    return np.maximum(0, x)

def max_pooling_2x2(x):
    """
    Max Pooling 2x2 con stride 2 (ventanas sin solape).
    x: array 2D (H, W) o 3D (H, W, C). Descarta filas/columnas sobrantes si H o W son impares.
    """
    if x.ndim == 3:
        # pooling por canal
        return np.stack([max_pooling_2x2(x[:, :, c]) for c in range(x.shape[2])], axis=2)

    H, W = x.shape
    H2, W2 = H // 2, W // 2          # nuevas dimensiones (mitad)
    x = x[:H2*2, :W2*2]              # descarta la fila/columna sobrante si son impares
    # Reorganiza en bloques 2x2: (H2, 2, W2, 2) y toma el max en las dimensiones de bloque
    return x.reshape(H2, 2, W2, 2).max(axis=(1, 3))

print("Funciones relu() y max_pooling_2x2() definidas.")

## 11. Aplicar el bloque: convolución → ReLU (x3) → Max Pooling

Para cada uno de los 3 kernels:
1. Se carga la imagen en **RGB** (se reutiliza `cargar_imagen` con `modo="rgb"`; si el archivo no existe, se reutiliza la imagen sintética ya generada).
2. Se aplica la **convolución** con `convolucion_3x3` (que ya soporta RGB canal por canal).
3. Se aplica **ReLU** sobre el resultado.
4. Se aplica **Max Pooling 2x2** al final.

Se guardan las 3 etapas de cada mapa para poder visualizar la evolución.

In [ ]:
# === BLOQUE CONVOLUCIONAL COMPLETO (EJERCICIO 3) ===
# Cargar la imagen en RGB (reutiliza la imagen sintética si imagen.jpg no existe)
try:
    img_rgb, _ = cargar_imagen(ruta_imagen, modo="rgb")
except FileNotFoundError:
    img_rgb = img if img.ndim == 3 else np.stack([img] * 3, axis=2)

print(f"Imagen RGB de entrada: {img_rgb.shape}")

mapas = {}   # nombre -> (después de conv, después de ReLU, después de pooling)

for nombre, k in kernels_ej3.items():
    conv = convolucion_3x3(img_rgb, k)        # 1) convolución
    act  = relu(conv)                          # 2) ReLU
    pool = max_pooling_2x2(act)                # 3) Max Pooling 2x2
    mapas[nombre] = (conv, act, pool)
    print(f"{nombre}")
    print(f"   tras convolución : {conv.shape}, rango [{conv.min():.2f}, {conv.max():.2f}]")
    print(f"   tras ReLU        : {act.shape}, rango [{act.min():.2f}, {act.max():.2f}]")
    print(f"   tras Max Pooling : {pool.shape}, rango [{pool.min():.2f}, {pool.max():.2f}]\n")

## 12. Visualización de los mapas de características

Se muestran:
- **Fila 1**: la imagen RGB original.
- **Fila 2**: los 3 mapas tras **convolución + ReLU** (mapas de características).
- **Fila 3**: los 3 mapas tras **Max Pooling 2x2** (la mitad del tamaño, con las activaciones más fuertes).

Los mapas se muestran en escala de grises (cada uno es un 'canal' producido por su kernel, igual que los feature maps de una CNN).

In [ ]:
# === VISUALIZACIÓN DE LOS MAPAS DE CARACTERÍSTICAS (EJERCICIO 3) ===

nombres = list(mapas.keys())

fig, ejes = plt.subplots(3, 3, figsize=(16, 13))

# Fila 1: imagen original RGB
for j in range(3):
    ejes[0, j].imshow(img_rgb, vmin=0, vmax=1)
    ejes[0, j].axis("off")
ejes[0, 1].set_title("Imagen original (RGB)", fontsize=13)

for j, nombre in enumerate(nombres):
    conv, act, pool = mapas[nombre]

    # Fila 2: convolución + ReLU (mapa de características)
    ejes[1, j].imshow(act, cmap="gray", vmin=0, vmax=act.max() if act.max() > 0 else 1)
    ejes[1, j].set_title(f"{nombre}\nConv + ReLU", fontsize=10)
    ejes[1, j].axis("off")

    # Fila 3: después de Max Pooling 2x2
    ejes[2, j].imshow(pool, cmap="gray", vmin=0, vmax=pool.max() if pool.max() > 0 else 1)
    ejes[2, j].set_title(f"{nombre.split(':')[0]}\nMax Pooling 2x2 ({pool.shape[0]}x{pool.shape[1]})", fontsize=10)
    ejes[2, j].axis("off")

plt.tight_layout()
plt.show()

### Conclusiones del ejercicio 3

- Este bloque replica manualmente la estructura de una **capa convolucional de una CNN**: varias convoluciones con kernels distintos → no-linealidad (ReLU) → reducción espacial (pooling).
- Cada kernel genera un **mapa de características** diferente: los Sobel resaltan bordes verticales/horizontales y el blur resalta zonas homogéneas.
- **ReLU** elimina las respuestas negativas (zonas oscuras en los mapas): solo sobreviven los píxeles donde el patrón del kernel está presente en la dirección positiva.
- **Max Pooling 2x2** reduce el tamaño a la mitad (H/2 x W/2), hace los mapas más robustos a pequeñas traslaciones y conserva solo la activación más fuerte de cada bloque.
- Si cada canal de salida se considerara un 'filtro aprendido', este conjunto de 3 mapas sería exactamente lo que produce la primera capa convolucional de una red como LeNet o VGG, pero con filtros fijados a mano en lugar de aprendidos.

---

# Ejercicio 4: Segmentación de objetos con YOLOv8 (mascotas, personas y elementos de vestir)

En los ejercicios anteriores los **kernels se definían a mano**. Ahora usamos un modelo de **deep learning ya entrenado** para hacer **segmentación de instancias**: detectar cada objeto de la imagen y pintar una **máscara píxel a píxel** sobre él.

**¿Por qué YOLOv8 y no VGG?**
- **VGG** es una red *clasificadora* de imágenes (dice 'esto es un perro'), pero **no segmenta**: no devuelve máscaras ni posiciones.
- **YOLOv8-seg** sí lo hace: en una sola pasada detecta los objetos, les pone una caja, una clase y una **máscara de segmentación**. Es el modelo más práctico y disponible.

**Clases relevantes del dataset COCO (80 clases) para este ejercicio:**
- **Personas**: `person`
- **Mascotas / animales**: `cat`, `dog`, `bird`, `horse`, `sheep`, `cow`, `bear`, `zebra`, `giraffe`
- **Elementos de vestir / accesorios**: `backpack`, `umbrella`, `handbag`, `tie`, `suitcase`

> Nota: COCO **no tiene clases de ropa específica** (camisa, pantalón...). Las prendas quedan incluidas dentro de la máscara de `person`. Para segmentar prendas haría falta un modelo entrenado en un dataset como DeepFashion2.

**Flujo del ejercicio:** el usuario carga la imagen → el modelo la procesa → se pinta cada instancia con un color → se muestra la imagen segmentada y un resumen de lo detectado por categoría.

In [ ]:
# === INSTALACIÓN DE LA LIBRERÍA (EJERCICIO 4) ===
# Si ultralytics no está instalada, descomenta la siguiente línea y ejecuta esta celda:
# %pip install ultralytics

try:
    import ultralytics
    from ultralytics import YOLO
    print(f"ultralytics disponible (versión {ultralytics.__version__})")
except ImportError:
    raise ImportError("ultralytics NO está instalada. Ejecuta: %pip install ultralytics y vuelve a correr esta celda.")

## 13. Carga del modelo y de la imagen del usuario

- `yolov8n-seg.pt` es el modelo **nano** de segmentación de YOLOv8, entrenado sobre **COCO**. La primera vez que se ejecuta, la librería **descarga automáticamente los pesos** (~7 MB).
- El **usuario carga su imagen escribiendo la ruta** (Enter para usar `imagen.jpg`). En **Google Colab** también se puede usar `files.upload()` para subir la foto desde el computador.

In [ ]:
# === CARGA DEL MODELO Y DE LA IMAGEN DEL USUARIO (EJERCICIO 4) ===

# Modelo nano de segmentación de instancias (COCO, 80 clases)
# La primera ejecución descarga los pesos automáticamente
modelo = YOLO("yolov8n-seg.pt")
print("Modelo yolov8n-seg cargado.")

# === El usuario carga su imagen ===
# (en Google Colab también puedes subir la foto con: from google.colab import files; files.upload())
ruta_ej4 = input("Ruta de la imagen a segmentar (Enter = imagen.jpg): ").strip() or "imagen.jpg"

try:
    img_ej4, _ = cargar_imagen(ruta_ej4, "rgb")   # reutiliza la función del ejercicio 1
    print(f"Imagen cargada: {ruta_ej4} -> {img_ej4.shape}")
except FileNotFoundError:
    raise FileNotFoundError(f"No se encontró '{ruta_ej4}'. Escribe una ruta válida (o Enter para usar imagen.jpg).")

## 14. Función de segmentación: el algoritmo retorna la imagen segmentada

`segmentar_imagen(ruta)` hace todo el trabajo y **retorna** tres cosas:
1. `img`: la imagen original (RGB, `[0, 1]`).
2. `img_seg`: la **imagen segmentada**, con la máscara de cada objeto pintada de un color sobre la foto.
3. `detecciones`: la lista de objetos detectados (clase, categoría, confianza y caja).

Internamente:
- `modelo.predict(ruta, conf=...)` ejecuta la inferencia: devuelve **cajas**, **clases**, **confianzas** y **máscaras** por instancia.
- Cada máscara (`res.masks.data`, un tensor binario por objeto) se **redimensiona al tamaño de la foto** si hace falta y se **mezcla con un color** de una paleta sobre la imagen (transparencia 50%).

In [ ]:
# === FUNCIÓN DE SEGMENTACIÓN (EJERCICIO 4) ===

# Paleta de colores para pintar cada instancia
PALETA = np.array([[231, 76, 60], [46, 204, 113], [52, 152, 219], [241, 196, 15],
                   [155, 89, 182], [26, 188, 156], [230, 126, 34], [149, 165, 166]], dtype=np.float32)

# Categorías de verificación (clases de COCO)
PERSONAS = {"person"}
MASCOTAS = {"cat", "dog", "bird", "horse", "sheep", "cow", "elephant", "bear", "zebra", "giraffe"}
VESTIR   = {"backpack", "umbrella", "handbag", "tie", "suitcase"}

def categoria_de(nombre):
    if nombre in PERSONAS: return "Persona"
    if nombre in MASCOTAS: return "Mascota/Animal"
    if nombre in VESTIR:   return "Vestir/Accesorio"
    return "Otro"

def segmentar_imagen(ruta, confianza=0.5):
    """
    Aplica YOLOv8-seg a la imagen en `ruta`.
    Retorna (img_original, img_segmentada, detecciones).
    """
    resultados = modelo.predict(source=ruta, conf=confianza, verbose=False)
    res = resultados[0]

    img, _ = cargar_imagen(ruta, "rgb")            # imagen original RGB en [0,1]
    H, W = img.shape[:2]
    img_seg = img.copy()                           # aquí se pintan las máscaras
    detecciones = []

    if res.masks is not None and len(res.boxes) > 0:
        for i, m in enumerate(res.masks.data):
            masc = m.cpu().numpy()
            if masc.shape != (H, W):               # la máscara viene en la resolución interna del modelo
                masc = np.asarray(Image.fromarray((masc * 255).astype(np.uint8))
                                  .resize((W, H), Image.NEAREST)) > 127
            color = PALETA[i % len(PALETA)] / 255.0
            img_seg[masc] = 0.5 * img_seg[masc] + 0.5 * color   # mezcla 50/50 con la foto

            nombre = res.names[int(res.boxes.cls[i])]
            detecciones.append({
                "clase": nombre,
                "categoria": categoria_de(nombre),
                "confianza": float(res.boxes.conf[i]),
                "caja": [float(v) for v in res.boxes.xyxy[i].tolist()]
            })
    else:
        print("No se detectaron objetos con la confianza indicada.")

    return img, img_seg, detecciones

# === Ejecutar la segmentación ===
img_ej4, img_seg_ej4, dets_ej4 = segmentar_imagen(ruta_ej4, confianza=0.5)

print(f"Objetos detectados: {len(dets_ej4)}")
for d in dets_ej4:
    print(f"   - {d['clase']:12s} | categoria: {d['categoria']:17s} | confianza: {d['confianza']:.2f}")

## 15. Visualización: imagen original vs. imagen segmentada

Subplot con la **foto cargada por el usuario** y la **imagen segmentada** retornada por el algoritmo, con las cajas y etiquetas (clase + confianza) de cada objeto detectado. Además se **guarda el resultado** en `imagen_segmentada.jpg`.

In [ ]:
# === VISUALIZACIÓN DEL RESULTADO (EJERCICIO 4) ===

fig, ejes = plt.subplots(1, 2, figsize=(14, 7))

ejes[0].imshow(img_ej4)
ejes[0].set_title("Imagen original (cargada por el usuario)")
ejes[0].axis("off")

ejes[1].imshow(img_seg_ej4)
for d in dets_ej4:
    x1, y1, x2, y2 = d["caja"]
    rect = plt.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, color="yellow", linewidth=2)
    ejes[1].add_patch(rect)
    ejes[1].text(x1, max(y1 - 6, 12), f"{d['clase']} {d['confianza']:.2f}",
                 color="yellow", fontsize=10, backgroundcolor="black")
ejes[1].set_title("Imagen segmentada (YOLOv8-seg)")
ejes[1].axis("off")

plt.tight_layout()
plt.show()

# Guardar la imagen segmentada que retorna el algoritmo
Image.fromarray((img_seg_ej4 * 255).astype(np.uint8)).save("imagen_segmentada.jpg")
print("Imagen segmentada guardada como 'imagen_segmentada.jpg'")

## 16. Verificación por categorías: mascotas, personas y elementos de vestir

Se agrupan las detecciones en las categorías de interés del enunciado para **verificar** qué contiene la fotografía. Prueba con fotos de:
- **Mascotas** (perros, gatos...) → aparecerán como `dog`, `cat`, etc.
- **Personas** → aparecerán como `person` (con su ropa incluida en la máscara).
- **Elementos de vestir / accesorios** → `backpack`, `handbag`, `tie`, `umbrella`, `suitcase`.
- Todo lo demás (sillas, coches, tazas...) se lista como **Otro**.

In [ ]:
# === VERIFICACIÓN POR CATEGORÍAS (EJERCICIO 4) ===

categorias = {"Persona": [], "Mascota/Animal": [], "Vestir/Accesorio": [], "Otro": []}
for d in dets_ej4:
    categorias[d["categoria"]].append(d)

print(f"Resumen de la foto '{ruta_ej4}':")
print("=" * 50)
for cat, lista in categorias.items():
    if lista:
        print(f"\n{cat} ({len(lista)} objeto(s)):")
        for d in lista:
            print(f"   - {d['clase']} (confianza: {d['confianza']:.2f})")

if not dets_ej4:
    print("No se detectaron objetos. Prueba con otra foto o baja la confianza (conf).")
else:
    hay_persona = len(categorias["Persona"]) > 0
    hay_mascota = len(categorias["Mascota/Animal"]) > 0
    hay_vestir = len(categorias["Vestir/Accesorio"]) > 0
    print("\nVerificación:")
    print(f"   Personas en la foto:          {'SI' if hay_persona else 'NO'}")
    print(f"   Mascotas/animales en la foto: {'SI' if hay_mascota else 'NO'}")
    print(f"   Vestir/accesorios en la foto: {'SI' if hay_vestir else 'NO'}")

### Conclusiones del ejercicio 4

- Con **YOLOv8-seg** la segmentación es automática: no se definen kernels a mano, sino que el modelo **aprendió** los filtros durante su entrenamiento sobre COCO (aquí se cierra el ciclo con los ejercicios 1 y 3: las primeras capas de estas redes hacen exactamente convoluciones 3x3 + ReLU + pooling).
- El algoritmo **retorna la imagen segmentada**: cada instancia detectada lleva su máscara de color, su caja y su etiqueta con la confianza.
- La verificación por categorías confirma el contenido de la foto: **personas**, **mascotas/animales** y **elementos de vestir/accesorios**.
- **Limitación importante**: COCO no incluye clases de ropa específica (camisa, pantalón, zapatos); esas prendas solo aparecen dentro de la máscara de `person` o como accesorios (`backpack`, `handbag`, `tie`). Para segmentar prendas habría que re-entrenar o usar un modelo con un dataset como DeepFashion2.
- El umbral `conf` (confianza) controla el balance entre **detectar más objetos** (bajo, más falsos positivos) y **detectar solo objetos seguros** (alto, puede omitir objetos difíciles).

---

# Fine-tuning del modelo de segmentación (mejorar la práctica anterior)

El `yolov8n-seg.pt` del Ejercicio 4 fue entrenado sobre **COCO**, por lo que reconoce solo sus 80 clases. El **fine-tuning** (ajuste fino) permite **mejorar la segmentación** para un dominio concreto (por ejemplo: una mascota específica, prendas de vestir, herramientas del laboratorio):

1. Se parte del modelo **pre-entrenado** (transfer learning): conserva todo lo aprendido en COCO.
2. Se re-entrena con un **dataset pequeño propio** (100–300 fotos anotadas bastan, gracias al transfer learning).
3. Solo se **ajustan suavemente los pesos** (learning rate pequeño) para no olvidar lo aprendido.

**Formato del dataset (YOLO-segmentación):**
- `dataset_ft/train/images/` y `dataset_ft/val/images/` → las fotos.
- `dataset_ft/train/labels/` y `dataset_ft/val/labels/` → un `.txt` por foto con un **polígono por objeto**: `clase x1 y1 x2 y2 x3 y3 ...` (coordenadas normalizadas 0–1).
- `dataset_ft/data.yaml` → define rutas, número de clases (`nc`) y sus nombres.

**Anotación:** herramientas como **Roboflow**, **CVAT** o **LabelMe** permiten dibujar los polígonos y exportar directamente en formato YOLO. Se recomienda entrenar en **Google Colab (GPU)** para mayor velocidad.

In [ ]:
# === VERIFICAR LIBRERÍAS PARA EL FINE-TUNING ===
# El fine-tuning usa la misma librería ultralytics del Ejercicio 4
# (si falta: %pip install ultralytics)

try:
    from ultralytics import YOLO
    print("ultralytics lista para el fine-tuning.")
except ImportError:
    raise ImportError("Instala ultralytics primero: %pip install ultralytics")

## 17. Preparar la estructura del dataset (formato YOLO)

Esta celda crea las carpetas y el `data.yaml` de ejemplo. **Ajusta `nc` y `names` a tus clases** y copia tus fotos anotadas en las carpetas correspondientes.

In [ ]:
# === CREAR ESTRUCTURA DEL DATASET PARA FINE-TUNING ===
import os
from pathlib import Path

# === AJUSTA ESTO A TUS CLASES ===
CLASES_FT = ["mi_mascota", "persona"]   # ej: clases del dominio propio
EPOCHS_FT = 50                            # épocas de fine-tuning

base = Path("dataset_ft")
for sub in ["train/images", "train/labels", "val/images", "val/labels"]:
    (base / sub).mkdir(parents=True, exist_ok=True)

# data.yaml en formato YOLO
yaml_text = (
    f"path: {base.resolve()}\n"
    "train: train/images\n"
    "val: val/images\n"
    f"nc: {len(CLASES_FT)}\n"
    f"names: {CLASES_FT}\n"
)
(base / "data.yaml").write_text(yaml_text, encoding="utf-8")

print("Estructura creada:")
for p in sorted(base.rglob("*")):
    print("  ", p)
print("\nCopie aqui sus fotos (images/) y sus anotaciones .txt (labels/).")
print("Recomendacion: dividir ~80% train, ~20% val.")

## 18. Entrenamiento (fine-tuning) y validación

`model.train()` parte de los pesos de COCO y **re-entrena** con el dataset propio. Hiperparámetros clave del fine-tuning:
- `data`: ruta del `data.yaml`.
- `epochs`: nº de pasadas (20–100 típico en fine-tuning).
- `imgsz=640`: tamaño de entrada.
- `lr0=0.001`: learning rate **pequeño** para ajustar sin olvidar lo aprendido.
- `batch`: según la memoria disponible (8–16 en Colab).

Después de entrenar, `model.val()` evalúa sobre el conjunto de validación (métricas **mAP50** y **mAP50-95** de máscara). El modelo ajustado queda en `runs/segment/train*/weights/best.pt` y se usa igual que en el Ejercicio 4.

In [ ]:
# === FINE-TUNING (EJECUTAR SOLO CON EL DATASET YA ANOTADO) ===

# Cargar el modelo pre-entrenado de segmentación (punto de partida del fine-tuning)
modelo_ft = YOLO("yolov8n-seg.pt")

# Re-entrenar (transfer learning) con el dataset propio
resultados_ft = modelo_ft.train(
    data="dataset_ft/data.yaml",
    epochs=EPOCHS_FT,
    imgsz=640,
    batch=8,
    lr0=0.001,           # learning rate pequeño: ajuste fino sin olvidar COCO
    name="finetune_seg"
)

# Validar el modelo ajustado (mAP de máscara)
metricas = modelo_ft.val()
print("mAP50 (mask):   ", metricas.seg.map50)
print("mAP50-95 (mask):", metricas.seg.map)

# === Usar el modelo mejorado ===
# modelo = YOLO("runs/segment/finetune_seg/weights/best.pt")
# img, seg, dets = segmentar_imagen(ruta_ej4)   # ahora segmenta mejor el dominio propio

---

# Ejercicio adicional 1: Tracker tipo esqueleto de la mano con la cámara (entorno local)

Se usa **MediaPipe Hands**, que internamente emplea **redes convolucionales (CNN)** para detectar la palma y predecir **21 puntos clave (landmarks)** de cada mano, con sus conexiones (el "esqueleto"). El video en vivo se captura con **OpenCV** desde la **cámara integrada** del equipo.

> **Importante:** este ejercicio es para **entorno local** (necesita webcam). En Google Colab **no funciona**. Instala primero: `%pip install mediapipe opencv-python`.

In [ ]:
# === INSTALACIÓN (solo la primera vez) ===
# %pip install mediapipe opencv-python

import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_styles = mp.solutions.drawing_styles

print(f"OpenCV {cv2.__version__} y MediaPipe {mp.__version__} listos.")

## Tracker de la mano en vivo

Ejecuta la siguiente celda y **muestra tu mano a la cámara**: verás el esqueleto con los 21 puntos (puntas de los dedos, nudillos y muñeca) dibujado en tiempo real. **Cierra la ventana o presiona `q`** para terminar y liberar la cámara.

In [ ]:
# === TRACKER DE ESQUELETO DE LA MANO (CÁMARA EN VIVO, ENTORNO LOCAL) ===

# Cámara integrada (índice 0). Si hay varias cámaras, prueba 1, 2...
camara = cv2.VideoCapture(0, cv2.CAP_DSHOW)
if not camara.isOpened():
    raise RuntimeError("No se pudo abrir la cámara. Verifica que exista y esté permitida.")

try:
    # CNN de MediaPipe: detecta la palma y regresa 21 landmarks por mano
    with mp_hands.Hands(static_image_mode=False,       # video (con seguimiento)
                        max_num_hands=2,               # hasta 2 manos
                        min_detection_confidence=0.7,
                        min_tracking_confidence=0.5) as manos:

        while camara.isOpened():
            ok, fotograma = camara.read()
            if not ok:
                break

            fotograma = cv2.flip(fotograma, 1)                       # efecto espejo
            rgb = cv2.cvtColor(fotograma, cv2.COLOR_BGR2RGB)         # MediaPipe usa RGB
            resultado = manos.process(rgb)

            if resultado.multi_hand_landmarks:
                for mano in resultado.multi_hand_landmarks:
                    # Dibuja los 21 puntos y las conexiones (esqueleto)
                    mp_drawing.draw_landmarks(
                        fotograma, mano, mp_hands.HAND_CONNECTIONS,
                        mp_styles.get_default_hand_landmarks_style(),
                        mp_styles.get_default_hand_connections_style())

            cv2.putText(fotograma, "'q' para salir", (10, 30),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)
            cv2.imshow("Tracker de mano (CNN + MediaPipe)", fotograma)

            if cv2.waitKey(1) & 0xFF == ord('q'):
                break
finally:
    camara.release()
    cv2.destroyAllWindows()

print("Tracker finalizado y cámara liberada.")

---

# Ejercicio adicional 2: Detección de objetos en un video con YOLO

El **usuario carga un video** al script y YOLO lo procesa **fotograma a fotograma**:

1. `cv2.VideoCapture` abre el video (formatos `.mp4`, `.avi`, `.mov`...).
2. Para cada fotograma, `YOLO('yolov8n.pt')` (modelo de **detección**, 80 clases COCO) dibuja cajas y etiquetas con `res.plot()`.
3. Todos los fotogramas anotados se escriben en un **video de salida** con `cv2.VideoWriter` (`video_procesado.mp4`).
4. Se acumula un **conteo de detecciones por clase** en todo el video y se muestran algunos fotogramas de muestra.

> Consejo: usa videos cortos (5–20 s) para la prueba en clase; el procesamiento en CPU tarda según la resolución y el número de fotogramas.

In [ ]:
# === DETECCIÓN DE OBJETOS EN VIDEO (EL USUARIO CARGA EL VIDEO) ===
from collections import Counter

# === El usuario carga su video ===
ruta_video = input("Ruta del video (Enter = video.mp4): ").strip() or "video.mp4"

cap = cv2.VideoCapture(ruta_video)
if not cap.isOpened():
    raise FileNotFoundError(f"No se pudo abrir el video '{ruta_video}'. Verifica la ruta y el formato.")

fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
ancho = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
alto = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
print(f"Video: {ancho}x{alto} @ {fps:.1f} fps | {total} fotogramas")

# Modelo de DETECCIÓN (80 clases COCO). La primera vez descarga los pesos (~6 MB)
modelo_video = YOLO("yolov8n.pt")

# Video de salida con las detecciones dibujadas
salida = cv2.VideoWriter("video_procesado.mp4",
                         cv2.VideoWriter_fourcc(*"mp4v"), fps, (ancho, alto))

conteo_clases = Counter()   # detecciones acumuladas en todo el video
muestras = []               # fotogramas anotados para mostrar después
n = 0

while cap.isOpened():
    ok, fotograma = cap.read()
    if not ok:
        break

    res = modelo_video.predict(fotograma, conf=0.4, verbose=False)[0]
    anotado = res.plot()                                   # fotograma con cajas y etiquetas (BGR)
    salida.write(anotado)                                  # se guarda en el video de salida

    for c in res.boxes.cls:
        conteo_clases[res.names[int(c)]] += 1

    # Guarda hasta 4 fotogramas de muestra repartidos en el video
    if len(muestras) < 4 and n % max(total // 4, 1) == 0:
        muestras.append(cv2.cvtColor(anotado, cv2.COLOR_BGR2RGB))

    n += 1
    if n % 30 == 0:
        print(f"   procesados {n}/{total} fotogramas...")

cap.release()
salida.release()
print(f"\nVideo procesado y guardado como 'video_procesado.mp4' ({n} fotogramas).")

## Muestra de fotogramas procesados y resumen de detecciones

Subplot con algunos fotogramas anotados tomados a lo largo del video y el **total de detecciones por clase**.

In [ ]:
# === VISUALIZACIÓN DE MUESTRAS Y RESUMEN (EJERCICIO ADICIONAL 2) ===

if muestras:
    fig, ejes = plt.subplots(2, 2, figsize=(14, 9))
    for eje, muestra in zip(ejes.flat, muestras):
        eje.imshow(muestra)
        eje.axis("off")
    fig.suptitle("Fotogramas de muestra con detecciones YOLO", fontsize=13)
    plt.tight_layout()
    plt.show()
else:
    print("No se capturaron muestras (¿el video tiene fotogramas?).")

print("Detecciones acumuladas en todo el video (por clase):")
if conteo_clases:
    for clase, cantidad in conteo_clases.most_common():
        print(f"   - {clase:12s}: {cantidad}")
else:
    print("   No se detectó ningún objeto.")

### Conclusiones de los ejercicios adicionales

- **Fine-tuning:** con un dataset propio pequeño y `lr0` bajo, el modelo de segmentación se **especializa** en el dominio del usuario (mejorando la segmentación de la práctica anterior) sin perder lo aprendido en COCO. Las métricas `mAP50`/`mAP50-95` de máscara permiten comprobar la mejora de forma objetiva.
- **Tracker de la mano:** MediaPipe combina **dos CNN en cascada** (detector de palma + regresor de 21 landmarks) para lograr el esqueleto en tiempo real en CPU; las mismas ideas de convolución de los ejercicios 1 y 3 son la base de estas redes.
- **Detección en video:** YOLO procesa cada fotograma de forma independiente y el resultado se **reescribe en un video de salida** con `cv2.VideoWriter`; el conteo acumulado por clase resume el contenido del video.
- Todas las herramientas comparten el mismo flujo: **imagen/video → CNN → predicciones → visualización**, cambiando solo la tarea (clasificar, detectar, segmentar o seguir puntos clave).